## MCUNet

Outputs

- `models/mcunet.tflite` normal (float32 input/output; TFLite)
- `models/mcunet_quantized.tflite` quantized (INT8 input/output, per-tensor; MicroFlow-compatible)

Keep `epochs` small if you only need artifacts quickly.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import tensorflow as tf

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

try:
    tf.get_logger().setLevel("ERROR")
except Exception:
    pass

REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OUT_TFLITE = MODELS_DIR / "mcunet.tflite"
OUT_TFLITE_QUANT = MODELS_DIR / "mcunet_quantized.tflite"

print("OUT_TFLITE:", OUT_TFLITE)
print("OUT_TFLITE_QUANT:", OUT_TFLITE_QUANT)

2026-03-30 12:56:33.188035: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 12:56:33.197162: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774868193.207297   48236 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774868193.210245   48236 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-30 12:56:33.221034: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

OUT_TFLITE: /home/nathan/Documents/ariel-microflow-ml/models/mcunet.tflite
OUT_TFLITE_QUANT: /home/nathan/Documents/ariel-microflow-ml/models/mcunet_quantized.tflite


In [2]:
def conv_block(x, filters, kernel_size, strides=1):
    # MicroFlow supports fused activations in CONV_2D (NONE/RELU/RELU6).
    return tf.keras.layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)

def depthwise_separable_block(x, pointwise_filters, strides=1):
    x = tf.keras.layers.DepthwiseConv2D(
        3,
        strides=strides,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)
    x = tf.keras.layers.Conv2D(
        pointwise_filters,
        1,
        padding="same",
        activation="relu",
        use_bias=True,
    )(x)
    return x

def build_mcunet() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(32, 32, 3), batch_size=1, name="input")
    x = conv_block(inputs, 16, 3, strides=1)
    x = depthwise_separable_block(x, 32, strides=2)
    x = depthwise_separable_block(x, 64, strides=2)
    x = depthwise_separable_block(x, 64, strides=1)

    # Replace GlobalAveragePooling (-> MEAN) with AveragePooling2D (-> AVERAGE_POOL_2D).
    logits_map = tf.keras.layers.Conv2D(10, 1, padding="same", activation=None, use_bias=True)(x)
    pooled = tf.keras.layers.AveragePooling2D(pool_size=(8, 8), strides=(8, 8), padding="valid")(logits_map)
    logits = tf.keras.layers.Reshape((10,), name="logits")(pooled)
    outputs = tf.keras.layers.Softmax(name="softmax")(logits)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="mcunet")

model = build_mcunet()
model.summary()

I0000 00:00:1774868194.622725   48236 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1190 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "mcunet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (1, 32, 32, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (1, 32, 32, 16)        │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (1, 16, 16, 16)        │           160 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (1, 16, 16, 32)        │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_1              │ (1, 8, 8, 32)          │           320 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (1, 8, 8, 64)          │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (1, 8, 8, 64)          │           640 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (1, 8, 8, 64)          │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (1, 8, 8, 10)          │           650 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (1, 1, 1, 10)          │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Reshape)                │ (1, 10)                │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Softmax)               │ (1, 10)                │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,034 (35.29 KB)

 Trainable params: 9,034 (35.29 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
epochs = 2
batch_size = 128

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")
x_train = (x_train.astype(np.float32) / 255.0)
x_test = (x_test.astype(np.float32) / 255.0)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    verbose=2,
)

print("Test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])

Epoch 1/2


I0000 00:00:1774868200.843683   52482 service.cc:148] XLA service 0x72a2b8010560 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774868200.843718   52482 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-03-30 12:56:40.874171: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774868201.048394   52482 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1774868204.542387   52482 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


352/352 - 13s - 36ms/step - accuracy: 0.1821 - loss: 2.1505 - val_accuracy: 0.2268 - val_loss: 2.0413
Epoch 2/2
352/352 - 3s - 9ms/step - accuracy: 0.2945 - loss: 1.8904 - val_accuracy: 0.3492 - val_loss: 1.7264
Test accuracy: 0.3555000126361847


In [4]:
def representative_data_gen():
    n = min(200, x_train.shape[0])
    for i in range(n):
        yield [x_train[i : i + 1].astype(np.float32)]

# Non-quantized (float32) TFLite export.
float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite_bytes = float_converter.convert()
OUT_TFLITE.write_bytes(float_tflite_bytes)
print("Wrote:", OUT_TFLITE, "bytes=", OUT_TFLITE.stat().st_size)

float_interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE), experimental_delegates=[])
float_interp.allocate_tensors()
print(
    "Float Input:",
    float_interp.get_input_details()[0]["dtype"],
    float_interp.get_input_details()[0]["shape"],
)
print(
    "Float Output:",
    float_interp.get_output_details()[0]["dtype"],
    float_interp.get_output_details()[0]["shape"],
)

# INT8 quantized TFLite export.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
converter.experimental_enable_resource_variables = True

for attr in ("experimental_new_quantizer", "_experimental_new_quantizer"):
    if hasattr(converter, attr):
        setattr(converter, attr, False)

tflite_bytes = converter.convert()
OUT_TFLITE_QUANT.write_bytes(tflite_bytes)
print("Wrote:", OUT_TFLITE_QUANT, "bytes=", OUT_TFLITE_QUANT.stat().st_size)

interp = tf.lite.Interpreter(model_path=str(OUT_TFLITE_QUANT), experimental_delegates=[])
interp.allocate_tensors()
print(
    "Quant Input:",
    interp.get_input_details()[0]["dtype"],
    interp.get_input_details()[0]["shape"],
)
print(
    "Quant Output:",
    interp.get_output_details()[0]["dtype"],
    interp.get_output_details()[0]["shape"],
)

per_channel = []
for td in interp.get_tensor_details():
    qp = td.get("quantization_parameters") or {}
    scales = qp.get("scales")
    if isinstance(scales, np.ndarray) and scales.size > 1:
        per_channel.append((td.get("name", "?"), int(scales.size)))

if per_channel:
    print("Warning: per-channel quantization detected (first 20):")
    for name, n in per_channel[:20]:
        print(" -", name, "scales=", n)
else:
    print("Per-channel quantization check: OK")

Saved artifact at '/tmp/tmpft79s1jh'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 32, 32, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  126046673110032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673111568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673110416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673110224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673110800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673109456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673111952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673111376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673112336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673110608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673113296: TensorSpec(

W0000 00:00:1774868219.548553   48236 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774868219.548579   48236 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-30 12:56:59.549153: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpft79s1jh
2026-03-30 12:56:59.549956: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-30 12:56:59.549967: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpft79s1jh
I0000 00:00:1774868219.555479   48236 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
2026-03-30 12:56:59.556558: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-30 12:56:59.596894: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpft79s1jh
2026-03-30 12:56:59.612835: I tensorflow/cc/saved_model/loader.cc:466] SavedModel 

Wrote: /home/nathan/Documents/ariel-microflow-ml/models/mcunet.tflite bytes= 40788
Float Input: <class 'numpy.float32'> [ 1 32 32  3]
Float Output: <class 'numpy.float32'> [ 1 10]
Saved artifact at '/tmp/tmpd113yony'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 32, 32, 3), dtype=tf.float32, name='input')
Output Type:
  TensorSpec(shape=(1, 10), dtype=tf.float32, name=None)
Captures:
  126046673110032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673111568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673110416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673110224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673110800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673109456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673111952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126046673111376: TensorSpec(shape=(), dtype=tf.resource, n

/home/nathan/Documents/ariel-microflow-ml/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774868220.166653   48236 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1774868220.166679   48236 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-03-30 12:57:00.166900: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpd113yony
2026-03-30 12:57:00.167496: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-03-30 12:57:00.167505: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpd113yony
2026-03-30 12:57:00.173112: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-03-30 12:57:00.205714: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel 